# Warsztat — przepływ agenta krok po kroku

Ten notebook uruchamia po kolei umiejętności z `src/skills/*` i pokazuje wynik
**każdego etapu** — dokładnie to, co robi `linear_agent/pipeline.py`, ale
interaktywnie, żeby zobaczyć outputy w przepływie.

> Wymaga skonfigurowanego `.env` (klucz LLM). Uruchamiaj z katalogu głównego repo.

In [ ]:
from src.loader import load_all
from src.sources import prepare_input_texts
from src.skills.file_description.main import generate_file_description
from src.skills.tasks.main import generate_tasks
from src.skills.make_task.main import make_task
from src.skills.strategy.main import generate_strategy
from src.skills.document.main import generate_document

GOAL = "Wygeneruj apelację z perspektywy obrony"
GOAL

## 0. Wczytanie akt

In [ ]:
documents = load_all()
print(f"Wczytano {len(documents)} dokumentów")
for d in documents:
    print(" -", d.filename)

## 1. `generate_file_description` — opis każdego dokumentu

Surowe akta → zwięzłe, ukierunkowane na cel opisy ("mapa" sprawy).

In [ ]:
described = [generate_file_description(doc, GOAL) for doc in documents]
for d in described:
    print(f"### {d.name}")
    print(d.title)
    print(d.description)
    print()

## 2. `generate_tasks` — plan analizy

Każdy krok wskazuje, **które** dokumenty są mu potrzebne (`input`).

In [ ]:
tasks = generate_tasks(GOAL, described)
for i, t in enumerate(tasks.thoughts, 1):
    print(f"[{i}] {t.action}")
    print(f"    dokumenty: {t.input}")
    print(f"    rozumowanie: {t.reasoning}")
    print()

## 3. `make_task` — wykonanie kroków na wybranych dokumentach

Sedno: `prepare_input_texts` daje krokowi **tylko** wskazane dokumenty (selektywny kontekst).

In [ ]:
task_outputs = []
for i, task in enumerate(tasks.thoughts, 1):
    sources_text = prepare_input_texts(documents, task.input)
    out = make_task(GOAL, task, sources_text)
    task_outputs.append(out)
    print(f"[{i}] {out.action}")
    print(out.output)
    print("-" * 70)

## 4. `generate_strategy` — wybór strategii

In [ ]:
strategy = generate_strategy(GOAL, task_outputs)
print("Definicja sukcesu:\n", strategy.definition_of_success)
print("\nMożliwe strategie:\n", strategy.strategies)
print("\nNajlepsza strategia:\n", strategy.best_strategy)

## 5. `generate_document` — gotowa apelacja

In [ ]:
document = generate_document(GOAL, strategy, task_outputs)
print(document.tekst)

## (opcjonalnie) Ewaluacja

Pokrywalność wygenerowanej apelacji względem `data/eval.json`.

In [ ]:
from src.eval.coverage import evaluate

result = evaluate(document.tekst)
print(f"Pokryte: {result.covered}/{result.total} = {result.score:.0%}\n")
for r in result.results:
    print(("✅" if r.covered else "❌"), r.issue[:80])